# Fraud Detection Assignment

This notebook is part of a real-world anomaly detection assignment.
The goal of this notebook is to understand the dataset and perform
initial data exploration before building models.

## Dataset Design 

Instead of randomly labeling transactions as fraud, I wanted the dataset
to reflect realistic behavioral patterns seen in payment fraud. 

The idea was to design fraud in a way that:
- Is not trivially separable
- Forces models to reason across multiple features
- Creates natural false positives and edge cases

This also allows meaningful comparison between anomaly detection models,
beyond just accuracy numbers.


## Problem Constraints and Assumptions

To simulate real-world fintech deployment conditions, the following
constraints were intentionally assumed throughout this assignment:

- **False positive cost is significantly higher than false negative cost**
  (approximately 10×), as incorrect fraud flags disrupt legitimate customers
  and incur operational overhead.

- **Fraud prevalence is low (<2%)**, reflecting realistic payment systems
  where fraud is rare and highly imbalanced.

- **Inference latency must remain low (<100 ms per transaction)** to support
  real-time payment authorization scenarios.

These constraints directly influenced model selection, evaluation metrics,
and threshold interpretation decisions.


In [3]:
import numpy as np
import pandas as pd

np.random.seed(42)

num_transactions = 100000

data = pd.DataFrame({
    "transaction_id": [f"TXN_{i:08d}" for i in range(num_transactions)],
    "customer_id": np.random.randint(1, 5001, num_transactions),
    "amount": np.random.exponential(scale=2500, size=num_transactions),
    "hour": np.random.randint(0, 24, num_transactions),
    "day_of_week": np.random.randint(0, 7, num_transactions),
    "distance_from_home": np.random.exponential(scale=8, size=num_transactions),
    "merchant_category": np.random.choice(
        ["grocery", "electronics", "restaurant", "fuel", "luxury"],
        size=num_transactions,
        p=[0.35, 0.2, 0.25, 0.15, 0.05]
    )
})

In [4]:
fraud_probability = (
    (data["amount"] > 15000).astype(int) * 0.35 +
    (data["distance_from_home"] > 25).astype(int) * 0.30 +
    (data["hour"].isin([0, 1, 2, 3])).astype(int) * 0.20 +
    (data["merchant_category"] == "luxury").astype(int) * 0.40
)

noise = np.random.uniform(0, 0.15, num_transactions)

data["fraud_score"] = fraud_probability + noise
data["is_fraud"] = (data["fraud_score"] > 0.6).astype(int)

data.drop(columns=["fraud_score"], inplace=True)


## Fraud Pattern Design Rationale

Fraud labels were not assigned randomly. Instead, multiple weak signals
were combined to simulate realistic fraud behavior:

- High transaction amounts increase fraud likelihood but are not sufficient alone
- Large distance from home reflects potential card misuse
- Late-night transactions add contextual risk
- Luxury merchants were intentionally given higher fraud influence

Noise was added to ensure fraud is probabilistic rather than deterministic.
This creates overlapping regions between fraud and legitimate transactions,
making the task non-trivial.


In [5]:
data.to_csv("../data/transactions.csv", index=False)
print("Updated dataset with intentional fraud patterns saved")

Updated dataset with intentional fraud patterns saved


In [6]:
df = pd.read_csv("../data/transactions.csv")
df.head()

,transaction_id,customer_id,amount,hour,day_of_week,distance_from_home,merchant_category,is_fraud
0,TXN_00000000,861,321.657107,12,6,8.349037,restaurant,0
1,TXN_00000001,3773,1944.835714,18,2,4.499823,grocery,0
2,TXN_00000002,3093,14551.485133,11,2,18.237703,grocery,0
3,TXN_00000003,467,977.794274,6,1,2.069479,restaurant,0
4,TXN_00000004,4427,106.712706,20,1,13.467285,restaurant,0


In [7]:
df["is_fraud"].value_counts(normalize=True)

is_fraud
0    0.98743
1    0.01257
Name: proportion, dtype: float64

### Dataset Specification 

The assignment mentions an approximate fraud rate of 2%.
In my synthetic dataset, the observed fraud rate is around 1.26%.

This deviation results from combining multiple weak fraud signals 
rather than enforcing a fixed class ratio. 